# 5b – Initial-Condition Analysis
## BruteForce vs JRA55-FOSIRL coupled restart files

**Purpose:** identify and quantify initial-state differences and physical imbalances that could
explain drift diagnosed later in **5a_refactor_drift_analysis.ipynb**. This notebook uses only
time-zero coupled restart files; it does not diagnose forecast drift itself. The sign convention is
$\Delta X_0 = X_{0,\,JRA55\text{-}FOSIRL} - X_{0,\,BruteForce}$.

### Recommended progression

1. Run the pilot date (`1980-05-01-00000`) end-to-end.
2. Establish file, timestamp, grid/mesh, cell-ordering, dimension, coordinate, mask/fraction, and unit integrity.
3. Separate metadata-only changes from physical-state changes and confirm atmospheric-member controls.
4. Quantify native-grid differences with weighted mean, RMSE, MAD, extrema, percentiles, threshold exceedance, pattern correlation, and NRMSE.
5. Examine conserved inventories, vertical structure, curated spatial fingerprints, distributions, and time-zero physical consistency.
6. Scale to all May and November starts and summarize recurrence, seasonality, sign consistency, spread, and exceptional starts.

> ℹ️ A checksum difference alone may result from metadata.
> A matching checksum proves byte identity; a checksum mismatch does not prove a physical difference.
> Cross-component checks are valid only on a shared grid or after applying the model's mapping weights.
> Conserved inventories require verified units plus area/volume, layer thickness, masks, and fractions; unweighted sums are never labeled as inventories.


In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

import sys
import esp_lab

# Development-checkout fallback: a clean install exposes ``workflows`` directly.
REPO_ROOT = Path(esp_lab.__file__).resolve().parent.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from workflows.diagnostics.initial_conditions import check_physical_consistency as physical_consistency_workflow
from workflows.diagnostics.initial_conditions import compare_netcdf_structure as structure_comparison_workflow
from workflows.diagnostics.initial_conditions import compute_ic_statistics as statistics_workflow
from workflows.diagnostics.initial_conditions import config as ic_config
from workflows.diagnostics.initial_conditions import inventory_and_hash as inventory_workflow
from workflows.diagnostics.initial_conditions import plot_component_differences as plotting_workflow
from workflows.diagnostics.initial_conditions import summarize_campaign as campaign_summary_workflow

from esp_lab.diagnostics import (
    ICConfig, ExperimentPair, ComponentSpec, DEFAULT_COMPONENTS,
    AuditResult, ic_variable_stats, aggregate_campaign_stats,
    check_atm_surface_vs_land, check_ocean_ice_consistency,
    write_manifest_json,
    discover_start_dates, match_start_dates,
    build_file_manifest, write_audit_csv, load_audit_csv,
    open_restart_file,
)
from esp_lab.utils.dask_utils import DaskConfig, get_cluster_client, close_cluster

load_config = ic_config.load_config
build_ic_config = ic_config.build_ic_config
IC_DIR = Path(ic_config.__file__).resolve().parent
print('ESP-Lab IC analysis imports OK')


ESP-Lab IC analysis imports OK


## Dask Setup

In [2]:
machine_env = os.environ.get('CLUSTER_TYPE', 'local')

dask_cfg = DaskConfig(
    cluster_type=machine_env,
    workers=8,
    cores=4,
    memory='16GB',
    walltime='04:00:00',
)

cluster, client = get_cluster_client(dask_cfg)
print(client)



!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
This can easily exceed memory/CPU limits and crash your Jupyter kernel.
We highly recommend launching Jupyter on an Exclusive Perlmutter Compute Node.
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
<Client: 'tcp://127.0.0.1:42433' processes=4 threads=4, memory=14.90 GiB>


## User Configuration

**All** parameters are defined here.  Edit this cell; no other cell below
needs to change.


In [3]:
# ── Load from config.yaml ───────────────────────────────────────
CONFIG_PATH = IC_DIR / 'config.yaml'
cfg_dict    = load_config(CONFIG_PATH)

# ── Pilot mode toggle ────────────────────────────────────────────
# Set PILOT_ONLY = False to process all start dates.
PILOT_ONLY = True
PILOT_DATE = '1980-05-01-00000'
FORCE_RECOMPUTE = False  # True only when source IC files have changed
REQUIRE_REVIEWED_THRESHOLDS = False  # Set True for publication/campaign production

# ── Build ICConfig ───────────────────────────────────────────────
ic_cfg = build_ic_config(cfg_dict, pilot_only=PILOT_ONLY)
if PILOT_ONLY:
    ic_cfg.pilot_date = PILOT_DATE

print('Active dates :', ic_cfg.active_dates)
print('Components   :', [c.name for c in ic_cfg.components])
print('ref root     :', ic_cfg.experiment_pair.ref_root)
print('test root    :', ic_cfg.experiment_pair.test_root)

# ── Output directories ───────────────────────────────────────────
OUT_ROOT     = Path(ic_cfg.output_root)
FIGURE_OUTDIR = Path(cfg_dict['output']['figure_outdir'])
MANIFEST_DIR = OUT_ROOT / 'manifests'
FC_DIR       = OUT_ROOT / 'file_comparison'
VS_DIR       = OUT_ROOT / 'variable_statistics'
MAPS_DIR     = FIGURE_OUTDIR / 'maps'
PROFILES_DIR = FIGURE_OUTDIR / 'profiles'
CS_DIR       = OUT_ROOT / 'campaign_summary'

threshold_cfg = cfg_dict.get('diagnostics', {}).get('meaningful_thresholds', {})
if REQUIRE_REVIEWED_THRESHOLDS and not any(threshold_cfg.get(c) for c in threshold_cfg):
    raise ValueError('Populate diagnostics.meaningful_thresholds in config.yaml before production.')
print('Threshold policy:', 'reviewed absolute' if REQUIRE_REVIEWED_THRESHOLDS else 'configured absolute, else 1% reference sigma')


Active dates : ['1980-05-01-00000']
Components   : ['atm', 'lnd', 'ocn', 'ice', 'rof', 'cpl']
ref root     : /global/cfs/cdirs/e3sm/zhan391/v3.LR.S2D.ENSINT/BruteForce
test root    : /global/cfs/cdirs/e3sm/zhan391/v3.LR.S2D.ENSINT/JRA55-FOSIRL
Threshold policy: configured absolute, else 1% reference sigma


## Step 1 — Inventory and Hash

Discover matched start-date directories, record restart timestamps, dimensions and paths, and compute SHA-256 checksums.

* Atmospheric EN00–EN09: all members hashed (control check).
* Non-atmospheric: one hash per start date.

**Expected result** for pilot: atmosphere files IDENTICAL, non-atmospheric TBD.


In [4]:
%%time

audit_df = inventory_workflow.run(
    config_path=CONFIG_PATH,
    pilot_only=PILOT_ONLY,
    single_date=PILOT_DATE if PILOT_ONLY else None,
    compute_hash=True,
    reuse_existing=not FORCE_RECOMPUTE,
    verbose=True,
)
IC_DATA_READY = not audit_df.empty
if IC_DATA_READY:
    display(audit_df.groupby(['component', 'status']).size().unstack(fill_value=0))
else:
    print('[STOP] No matched IC files were found. Verify the configured roots and active date.')


IC Analysis — Step 1: Inventory and Hash
  ref  : /global/cfs/cdirs/e3sm/zhan391/v3.LR.S2D.ENSINT/BruteForce
  test : /global/cfs/cdirs/e3sm/zhan391/v3.LR.S2D.ENSINT/JRA55-FOSIRL
  dates: ['1980-05-01-00000']
  hash : True

  [REUSE] 1980-05-01-00000 → /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/multimodel/initial_conditions/manifests/1980-05-01-00000.csv

  Combined manifest → /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/multimodel/initial_conditions/manifests/combined_manifest.json

--- Audit Summary ---
component  status   
atm        DIFFERENT    10
cpl        IDENTICAL     1
ice        DIFFERENT     1
lnd        IDENTICAL     1
ocn        DIFFERENT     1
rof        IDENTICAL     1



/global/u2/z/zhan391/code/ESP-Lab/workflows/diagnostics/initial_conditions/inventory_and_hash.py:143: UserWarning: match_start_dates: 2 date(s) exist only in test (e.g. 1980-01-01-00000); skipping.
  all_pairs = match_start_dates(ref_root, test_root)


status,DIFFERENT,IDENTICAL
component,,
atm,10,0
cpl,0,1
ice,1,0
lnd,0,1
ocn,1,0
rof,0,1


CPU times: user 87.1 ms, sys: 76.8 ms, total: 164 ms
Wall time: 2.11 s


### Atmospheric control check

In [5]:
if IC_DATA_READY:
    manifest_csv = MANIFEST_DIR / f"{ic_cfg.active_dates[0]}.csv"
    manifest = load_audit_csv(manifest_csv)
    atm_rows = manifest[manifest['component'] == 'atm']
    print('Atmospheric checksums:')
    print(atm_rows[['member', 'status', 'ref_sha256', 'test_sha256']].to_string(index=False))
else:
    print('[SKIP] Atmospheric control check: inventory is empty.')


Atmospheric checksums:
member    status                                                       ref_sha256                                                      test_sha256
  EN00 DIFFERENT 47910e15762b931403dd46acf56143d7096312ac229f83334b10bd3c6e133922 a93f8ca4a03754a6a73429c4e8d8feb334ceb37f5dfd10782e2061b03a3dc78c
  EN01 DIFFERENT a0c79798e2efdb158498fa523ead0b0c359652826bbd73e1eabdf5c12f747d69 02aadbf9f169489365cca991b9e5fea5250935150eb83f14ea9789042908d748
  EN02 DIFFERENT 422c8574a322ed05793e6af415cb46c00273f7296e90b84771b2f6c129b9ed54 7decbc26e6c300b43a411b7ad2b526933defdb55abdd830b7cd1ee2da114c6da
  EN03 DIFFERENT 467d38e6fb2570abbcb4a2772a8710b73bec7eb6f5d07e5488fc1d951c1c50e7 6b3d3e00937d6bc591651981bd0218025e1136aa37fffceda02392959a2e21ab
  EN04 DIFFERENT 6434e52ef6178a42e211ab61471015a90cd08f40dd776b14baa360f91e8cf811 7ea07eada60ffe85a3b2ecb7b8127662ed8b716b92c7bca4bbc8fb12833ba08a
  EN05 DIFFERENT b7f9669efa0ffa5887e5ccbefa7752b645ef90f8eab9c1b6a3647588edd3612b d7e2359c4f7d3

## Step 2 — NetCDF Structure Comparison

For every file flagged DIFFERENT, compare variable lists, dimension sizes, data types,
coordinate values/order, MPAS mesh descriptors, and physical metadata (including units).
Classify each variable as *physical*, *metadata*, or *unknown*; review the unknown class manually.

> A checksum difference from metadata alone does **not** mean the physical
> state differs.  This step identifies which variables to actually analyse.


In [ ]:
%%time

classif_df = (structure_comparison_workflow.run(
    config_path=CONFIG_PATH,
    pilot_only=PILOT_ONLY,
    single_date=PILOT_DATE if PILOT_ONLY else None,
    verbose=True,
) if IC_DATA_READY else pd.DataFrame())
if not IC_DATA_READY:
    print('[SKIP] Step 2: inventory is empty.')
if not classif_df.empty:
    display(classif_df.groupby(['component', 'classification', 'location'])
            .size().unstack(fill_value=0))
    schema_files = sorted(FC_DIR.glob('*_schema.csv'))
    schema_df = pd.concat([pd.read_csv(p) for p in schema_files], ignore_index=True) if schema_files else pd.DataFrame()
    integrity_cols = ['start_date', 'component', 'member', 'n_dim_mismatch', 'n_dtype_mismatch',
                      'n_coord_mismatch', 'n_variable_attr_mismatch']
    if not schema_df.empty:
        display(schema_df[[c for c in integrity_cols if c in schema_df]].sort_values(['start_date', 'component']))
    unknown = classif_df.query("classification == 'unknown'")
    if not unknown.empty:
        print(f'[REVIEW] {len(unknown)} unknown variables must be classified before production.')


IC Analysis — Step 2: NetCDF Structure Comparison

  Date: 1980-05-01-00000  (12 DIFFERENT/INCOMPATIBLE rows)
    [atm] common=130 only_ref=0 only_test=0 dim_mismatch=0 grid/coord_mismatch=0 unit/attr_mismatch=0
    [atm] common=130 only_ref=0 only_test=0 dim_mismatch=0 grid/coord_mismatch=0 unit/attr_mismatch=0
    [atm] common=130 only_ref=0 only_test=0 dim_mismatch=0 grid/coord_mismatch=0 unit/attr_mismatch=0
    [atm] common=130 only_ref=0 only_test=0 dim_mismatch=0 grid/coord_mismatch=0 unit/attr_mismatch=0
    [atm] common=130 only_ref=0 only_test=0 dim_mismatch=0 grid/coord_mismatch=0 unit/attr_mismatch=0
    [atm] common=130 only_ref=0 only_test=0 dim_mismatch=0 grid/coord_mismatch=0 unit/attr_mismatch=0
    [atm] common=130 only_ref=0 only_test=0 dim_mismatch=0 grid/coord_mismatch=0 unit/attr_mismatch=0
    [atm] common=130 only_ref=0 only_test=0 dim_mismatch=0 grid/coord_mismatch=0 unit/attr_mismatch=0
    [atm] common=130 only_ref=0 only_test=0 dim_mismatch=0 grid/coord_mism

location                  common
component classification        
atm       metadata            70
          physical            10
          unknown           1220
ice       metadata             2
          physical             5
          unknown             67
ocn       metadata             6
          physical            19
          unknown             65

,start_date,component,member,n_dim_mismatch,n_dtype_mismatch,n_coord_mismatch,n_variable_attr_mismatch
0,1980-05-01-00000,atm,EN00,0,0,0,0
1,1980-05-01-00000,atm,EN01,0,0,0,0
2,1980-05-01-00000,atm,EN02,0,0,0,0
3,1980-05-01-00000,atm,EN03,0,0,0,0
4,1980-05-01-00000,atm,EN04,0,0,0,0
5,1980-05-01-00000,atm,EN05,0,0,0,0
6,1980-05-01-00000,atm,EN06,0,0,0,0
7,1980-05-01-00000,atm,EN07,0,0,0,0
8,1980-05-01-00000,atm,EN08,0,0,0,0
9,1980-05-01-00000,atm,EN09,0,0,0,0


[REVIEW] 1352 unknown variables must be classified before production.
CPU times: user 1.73 s, sys: 748 ms, total: 2.48 s
Wall time: 8.8 s


: 

## Step 3 — Variable-Level IC Statistics

For every *physical* common variable in each DIFFERENT file, compute the native-grid diagnostic contract:
weighted mean difference, RMSE, MAD, maximum absolute difference, 5th/50th/95th percentiles,
meaningful-threshold exceedance fraction, pattern correlation, and
$\mathrm{NRMSE}=\mathrm{RMSE}(\Delta X_0)/\sigma_w(X_{0,reference})$.
NRMSE—not raw RMSE—is used for cross-variable ranking. The workflow also writes signed, relative,
standardized, and threshold-mask fields for curated fingerprints.


In [ ]:
%%time

stats_df = (statistics_workflow.run(
    config_path=CONFIG_PATH,
    pilot_only=PILOT_ONLY,
    single_date=PILOT_DATE if PILOT_ONLY else None,
    verbose=True,
) if IC_DATA_READY else pd.DataFrame())
if not IC_DATA_READY:
    print('[SKIP] Step 3: inventory is empty.')
if not stats_df.empty:
    display(
        stats_df.groupby('component')[['nrmse', 'rmse', 'mad', 'pattern_corr', 'frac_exceeding_threshold']]
        .agg(['mean', 'max']).round(4)
    )


IC Analysis — Step 3: Compute Variable-Level IC Statistics

  [1980-05-01-00000] atm EN00
    cosp_temp_bnds                            RMSE=0  MAD=0  r=1.000

  [1980-05-01-00000] atm EN01
    cosp_temp_bnds                            RMSE=0  MAD=0  r=1.000

  [1980-05-01-00000] atm EN02
    cosp_temp_bnds                            RMSE=0  MAD=0  r=1.000

  [1980-05-01-00000] atm EN03
    cosp_temp_bnds                            RMSE=0  MAD=0  r=1.000

  [1980-05-01-00000] atm EN04
    cosp_temp_bnds                            RMSE=0  MAD=0  r=1.000

  [1980-05-01-00000] atm EN05
    cosp_temp_bnds                            RMSE=0  MAD=0  r=1.000

  [1980-05-01-00000] atm EN06
    cosp_temp_bnds                            RMSE=0  MAD=0  r=1.000

  [1980-05-01-00000] atm EN07
    cosp_temp_bnds                            RMSE=0  MAD=0  r=1.000

  [1980-05-01-00000] atm EN08
    cosp_temp_bnds                            RMSE=0  MAD=0  r=1.000

  [1980-05-01-00000] atm EN09
    cosp_t

### Cross-variable ranking by NRMSE

In [ ]:
if not stats_df.empty:
    top = stats_df.nlargest(20, 'nrmse')[
        ['component', 'variable', 'units', 'nrmse', 'rmse', 'mad', 'max_abs_diff',
         'p05_diff', 'p50_diff', 'p95_diff', 'pattern_corr',
         'meaningful_threshold', 'threshold_source', 'frac_exceeding_threshold']
    ].reset_index(drop=True)
    display(top)


## Step 4 — Inventories, Vertical Structure, Spatial Fingerprints, and Distributions

The inventory audit reports absolute and percentage differences only when the weighting is physically
valid. Area-weighted native sums are useful candidates but are **not** automatically conserved inventories:
soil/snow/canopy water needs land area and fractions; ocean heat/freshwater needs layer volume and constants;
sea-ice area/volume needs concentration/thickness and ocean-cell area; atmospheric water/dry mass needs pressure
or layer mass. Confirm formulas and units before interpreting `integral_*` columns.

For scientifically selected land, ocean, ice, and river variables, the plotting workflow generates signed,
relative, standardized, and threshold-exceedance native-grid fingerprints; global histogram plus global, Arctic,
tropical, and Antarctic empirical CDFs when latitude is available; and
layer-wise weighted mean difference and RMSE. Sign changes in the mean profile expose vertical compensation.


In [ ]:
%%time

if not stats_df.empty:
    inventory_terms = r'h2osoi|soilwater|h2osno|swe|canopy|temperature|salinity|aice|iceconcentration|icethickness|river|storage'
    inventory_audit = stats_df[stats_df['variable'].str.contains(inventory_terms, case=False, regex=True, na=False)].copy()
    inventory_cols = ['start_date', 'component', 'variable', 'units', 'weighting',
                      'integral_ref', 'integral_test', 'integral_diff', 'integral_pct_diff']
    display(inventory_audit[[c for c in inventory_cols if c in inventory_audit]].sort_values(['component', 'variable']))
    print('[REVIEW] These are inventory candidates. Accept as conserved only after validating measures, fractions, units, and formulas.')

if IC_DATA_READY and not stats_df.empty:
    plotting_workflow.run(
        config_path=CONFIG_PATH,
        pilot_only=PILOT_ONLY,
        single_date=PILOT_DATE if PILOT_ONLY else None,
        top_n=15,  # bar charts are ranked by dimensionless NRMSE
        figure_outdir=FIGURE_OUTDIR,
        verbose=True,
    )
else:
    print('[SKIP] Step 4: no variable statistics are available.')


### Inline map viewer

In [ ]:
from IPython.display import Image, display as ipy_display

pilot_maps = sorted(MAPS_DIR.glob(f"{ic_cfg.active_dates[0]}_*_fingerprint.png")) if IC_DATA_READY else []
for p in pilot_maps[:6]:
    print(p.name)
    ipy_display(Image(str(p), width=800))


## Step 5 — Physical Consistency Checks

Time-zero checks run independently on each experiment, then their severity can be compared:

* atmosphere surface temperature vs land ground temperature;
* SST vs sea-ice concentration and local freezing plausibility;
* restart timestamp agreement across components.

A check is explicitly skipped when native grids differ; supply the model's mapping weights rather than
allowing coordinate alignment to masquerade as regridding. Extend the same framework, once variable names and
units are confirmed, to atmosphere humidity/surface conditions, snow state consistency, soil moisture bounds,
temperature/salinity ranges, canopy/surface water, and river/storage bounds.


In [ ]:
%%time

consistency_df = (physical_consistency_workflow.run(
    config_path=CONFIG_PATH,
    pilot_only=PILOT_ONLY,
    single_date=PILOT_DATE if PILOT_ONLY else None,
    verbose=True,
) if IC_DATA_READY else pd.DataFrame())
if not IC_DATA_READY:
    print('[SKIP] Step 5: inventory is empty.')
if not consistency_df.empty:
    display(consistency_df[['check_name', 'experiment', 'frac_inconsistent', 'notes']])


## Step 6 — Campaign Summary

Across all May and November starts, summarize the median and interquartile spread of RMSE/NRMSE,
May-minus-November contrast, recurrence and sign consistency, and the fraction of total RMSE contributed
by the largest start (an exceptional-start diagnostic).

> ℹ️ This step requires data from all start dates.  Set `PILOT_ONLY = False`
> in the User Configuration cell and re-run Steps 1–5 before running this cell.


In [ ]:
%%time

campaign_df = (campaign_summary_workflow.run(
    config_path=CONFIG_PATH,
    pilot_only=PILOT_ONLY,
    verbose=True,
) if IC_DATA_READY and not PILOT_ONLY else pd.DataFrame())
if PILOT_ONLY:
    print('[SKIP] Step 6 requires PILOT_ONLY = False and completed full-campaign diagnostics.')
elif not IC_DATA_READY:
    print('[SKIP] Step 6: inventory is empty.')
if not campaign_df.empty:
    display(campaign_df.nlargest(10, 'median_nrmse'))


### Campaign heatmap

In [ ]:
from IPython.display import Image, display as ipy_display

heatmap_path = CS_DIR / 'campaign_summary.png'
if heatmap_path.exists():
    ipy_display(Image(str(heatmap_path), width=900))
else:
    print(f'Not found: {heatmap_path} -- run Step 6 with full campaign first.')


## Shutdown

In [ ]:
close_cluster(cluster, client)
